# Day 56 · Exercise 4: Text Extraction

**What you'll build:** Implement `extract_text(content, content_type)` that returns plain text from either a UTF-8 text file or a PDF. This is what turns a binary upload into something an LLM can read and reason about.

## Setup (provided)

In [ ]:
import io
import pypdf


## Your Implementation

In [ ]:
def extract_text(content: bytes, content_type: str) -> str:
    """Extract plain text from uploaded file bytes.

    Args:
        content:      Raw file bytes.
        content_type: MIME type string (e.g. 'text/plain', 'application/pdf').
    Returns:
        Extracted text as a string.
        - For PDF ('pdf' in content_type): use pypdf.PdfReader on io.BytesIO(content),
          concatenate page.extract_text() for each page.
        - Otherwise: decode as UTF-8 with errors='replace'.
    """
    # TODO: if "pdf" in content_type:
    #     reader = pypdf.PdfReader(io.BytesIO(content))
    #     return "\n".join(p.extract_text() or "" for p in reader.pages)
    # return content.decode("utf-8", errors="replace")
    raise NotImplementedError


In [ ]:
def extract_text(content: bytes, content_type: str) -> str:
    if "pdf" in content_type:
        reader = pypdf.PdfReader(io.BytesIO(content))
        return "\n".join(p.extract_text() or "" for p in reader.pages)
    return content.decode("utf-8", errors="replace")


## Check Your Work

In [ ]:
_MINIMAL_PDF = b'%PDF-1.4\n1 0 obj\n<< /Type /Catalog /Pages 2 0 R >>\nendobj\n2 0 obj\n<< /Type /Pages /Kids [3 0 R] /Count 1 >>\nendobj\n3 0 obj\n<< /Type /Page /Parent 2 0 R /MediaBox [0 0 200 200]\n   /Contents 4 0 R /Resources << /Font << /F1 5 0 R >> >> >>\nendobj\n4 0 obj\n<< /Length 44 >>\nstream\nBT /F1 12 Tf 50 150 Td (Hello PDF) Tj ET\nendstream\nendobj\n5 0 obj\n<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>\nendobj\nxref\n0 6\n0000000000 65535 f \n0000000009 00000 n \n0000000058 00000 n \n0000000115 00000 n \n0000000274 00000 n \n0000000366 00000 n \ntrailer\n<< /Size 6 /Root 1 0 R >>\nstartxref\n450\n%%EOF\n'

def _run_checks():
    score = 0
    total = 5

    def _chk(n, ok, msg):
        nonlocal score
        print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
        if ok:
            score += 1

    try:
        text = extract_text(b"Hello, plain text!", "text/plain")
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: extract_text not implemented")
        print(f"\nScore: 0 / {total}")
        return

    _chk(1, isinstance(text, str) and "Hello" in text,
         f"plain text extracted correctly (got {text!r:.40})")

    # UTF-8 with replacement
    bad_bytes = b"caf\xe9"
    t2 = extract_text(bad_bytes, "text/plain")
    _chk(2, isinstance(t2, str),
         f"non-UTF-8 bytes → str (errors=replace) (got {t2!r})")

    # PDF extraction
    try:
        pdf_text = extract_text(_MINIMAL_PDF, "application/pdf")
        _chk(3, isinstance(pdf_text, str),
             f"PDF extraction returns str (got {type(pdf_text).__name__})")
        _chk(4, len(pdf_text) >= 0,   # pypdf may return empty for this minimal PDF
             "PDF extraction does not crash")
    except Exception as e:
        _chk(3, False, f"PDF extraction raised {type(e).__name__}: {e}")
        _chk(4, False, "skipped")

    # Unknown type falls back to UTF-8 decode
    t5 = extract_text(b"raw bytes", "application/octet-stream")
    _chk(5, isinstance(t5, str) and "raw" in t5,
         f"unknown type decoded as UTF-8 (got {t5!r})")

    print(f"\nScore: {score} / {total}")
    if score == total:
        print("🎉 Exercise complete!")

_run_checks()


## Bonus Challenge

Add Markdown support: if `content_type == 'text/markdown'` or the filename ends with `.md`, strip Markdown syntax before returning (remove `#`, `**`, `_`, `[`, `]`, `(`, `)` etc.) using a simple regex. Plain text is easier for the LLM to process than raw Markdown.

## Solution

<details>
<summary>Show solution</summary>

```python
def extract_text(content: bytes, content_type: str) -> str:
    if "pdf" in content_type:
        reader = pypdf.PdfReader(io.BytesIO(content))
        return "\n".join(p.extract_text() or "" for p in reader.pages)
    return content.decode("utf-8", errors="replace")
```

**Why this works:** The `"pdf" in content_type` check handles both
`application/pdf` and variants like `application/x-pdf`. `pypdf.PdfReader` accepts
a file-like `io.BytesIO` object, so you never need to write the PDF to disk.
Each `page.extract_text()` returns a string (or None for image-only pages — `or ""`
handles that). For everything else, `decode("utf-8", errors="replace")` gives a
string with replacement characters (U+FFFD) for invalid bytes, so it never raises.

</details>